In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import os
import warnings
import vectorbtpro as vbt

MIN_STOCKS = 25
TOP_QUANTILE = 0.80
MONTHS_PER_YEAR = 12
INIT_CASH = 1000000
COST_BPS = 20
FEE_RATE = COST_BPS / 10000


In [ ]:
BANK_TICKERS = {
    "ABB", "ACB", "BAB", "BID", "BVB", "CTG", "EIB", "HDB", "KLB", "LPB",
    "MBB", "MSB", "NAB", "NVB", "OCB", "PGB", "SGB", "SHB", "SSB", "STB",
    "TCB", "TPB", "VAB", "VBB", "VCB", "VIB", "VPB",
}
SECURITIES_TICKERS = {
    "AAS", "AGR", "APG", "APS", "ART", "BMS", "BSI", "BVS", "CSI", "CTS",
    "DSC", "DSE", "EVS", "FTS", "HAC", "HBS", "HCM", "IVS", "MBS", "ORS",
    "PSI", "SBS", "SHS", "SSI", "TCI", "TVB", "TVC", "TVS", "VCI", "VDS",
    "VFS", "VIG", "VIX", "VND", "WSS",
}
INSURANCE_TICKERS = {
    "ABI", "AIC", "BHI", "BIC", "BLI", "BMI", "BSH", "BVH", "MIG", "PGI",
    "PRE", "PTI", "PVI", "VNR",
}
OTHER_FINANCIAL_TICKERS = {"EVF", "IPA"}
FINANCIAL_PREFIX_EXCLUSIONS = ("E1VF", "FUE", "FUC")

FINANCIAL_TICKER_EXCLUSIONS = (
    BANK_TICKERS | SECURITIES_TICKERS | INSURANCE_TICKERS | OTHER_FINANCIAL_TICKERS
)

In [ ]:
raw_fund = pd.read_csv(r"D:\quant idea local\quant-idea\ranking\data_ranking\data_fund\fundamental.csv", index_col= 0)
raw_fund = raw_fund.rename(columns={'Mã':'ticker'})

In [ ]:
numeric_cols = ['MV', 'Quarter', 'Year', 'D.A', 'GEBT', 'ROE', 'EPS',
                'FCFF', 'GGP', 'GRev', 'EV.EBITDA', 'P.E', 'P.B',
                'FCF', 'LNG', 'LNT']

raw_fund[numeric_cols] = raw_fund[numeric_cols].apply(pd.to_numeric, errors='coerce')

In [ ]:
financial_mask = raw_fund['ticker'].isin(FINANCIAL_TICKER_EXCLUSIONS) | raw_fund['ticker'].str.startswith(FINANCIAL_PREFIX_EXCLUSIONS)
fund = raw_fund[~financial_mask].copy()
fund

,ticker,MV,Quarter,Year,D.A,GEBT,ROE,EPS,FCFF,GGP,GRev,EV.EBITDA,P.E,P.B,FCF,LNG,LNT
0,AAA,NaN,1,2008,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AAA,NaN,2,2008,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AAA,NaN,3,2008,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,AAA,NaN,4,2008,0.58,NaN,0.2564,NaN,-1.026268e+11,NaN,NaN,NaN,NaN,NaN,-1.026268e+11,0.2093,0.0921
8,AAA,NaN,1,2009,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61190,YEG,2.186543e+12,4,2024,0.23,1.7211,0.0893,639.45,-1.642559e+11,-0.2826,1.5204,56.16,17.83,1.52,-1.642559e+11,0.1665,0.1195
61192,YEG,2.493426e+12,1,2025,0.10,1.0123,0.0865,877.16,-2.808651e+11,1.9238,1.9633,35.70,14.82,1.24,-2.808651e+11,0.1703,0.1143
61194,YEG,2.407116e+12,2,2025,0.08,1.7671,0.0886,992.27,-2.268324e+11,2.0196,1.1804,20.60,12.65,1.18,-2.268324e+11,0.1645,0.1131
61196,YEG,2.349575e+12,3,2025,0.09,-0.6575,0.0715,806.13,-2.492897e+11,-0.4278,0.1331,23.62,15.20,1.14,-2.492897e+11,0.1362,0.0911


In [ ]:
fund.columns

Index(['ticker', 'MV', 'Quarter', 'Year', 'D.A', 'GEBT', 'ROE', 'EPS', 'FCFF',
       'GGP', 'GRev', 'EV.EBITDA', 'P.E', 'P.B', 'FCF', 'LNG', 'LNT'],
      dtype='object')

In [ ]:
fund = fund.rename(columns={
    "MV": 'mv',
    "Quarter": 'quarter',
    'Year': 'year',
    'D.A': 'd_a',
    'GEBT': 'ebt_growth',
    'ROE': 'roe',
    'EPS': 'eps',
    'FCFF': 'fcff',
    'GGP': 'gross_profit_growth',
    'GRev': 'revenue_growth',
    'EV.EBITDA': 'ev_ebitda',
    'P.E': 'pe',
    'P.B': 'pb',
    'FCF': 'fcf',
    'LNG': 'gross_margin',
    'LNT': 'net_margin'
})

In [ ]:
price = []
for ticker in sorted(fund['ticker'].dropna().unique()):
    path = f"D:/quant idea local/quant-idea/ranking/data_ranking/data_1d/{ticker}_1d.pkl"
    if os.path.exists(path):
        ticker_price = pd.read_pickle(path)
        ticker_price = ticker_price[['datetime', 'close']].copy()
        ticker_price['ticker'] = ticker
        price.append(ticker_price[['ticker', 'datetime', 'close']])

prices = pd.concat(price, ignore_index = True)
prices['date'] = pd.to_datetime(prices['datetime'], errors= 'coerce')
prices['close'] = pd.to_numeric(prices['close'], errors= 'coerce')
prices = prices.dropna(subset=['ticker', 'date', 'close']).sort_values(['ticker', 'date']).copy()
prices

,ticker,datetime,close,date
0,AAA,2013-01-02 07:00:00,3.830,2013-01-02 07:00:00
1,AAA,2013-01-03 07:00:00,3.724,2013-01-03 07:00:00
2,AAA,2013-01-04 07:00:00,3.777,2013-01-04 07:00:00
3,AAA,2013-01-07 07:00:00,3.751,2013-01-07 07:00:00
4,AAA,2013-01-08 07:00:00,3.751,2013-01-08 07:00:00
...,...,...,...,...
983865,YEG,2025-12-25 07:00:00,12.850,2025-12-25 07:00:00
983866,YEG,2025-12-26 07:00:00,12.450,2025-12-26 07:00:00
983867,YEG,2025-12-29 07:00:00,12.400,2025-12-29 07:00:00
983868,YEG,2025-12-30 07:00:00,12.450,2025-12-30 07:00:00


In [ ]:
# Build accounting-derived variables used by the signal.
if fund['eps'].notna().sum() > 0:
    eps_lag = fund.groupby('ticker')['eps'].shift(4)
    fund['eps_growth'] = 2 * (fund['eps'] - eps_lag) / (fund['eps'].abs() + eps_lag.abs()).replace(0, np.nan)
else:
    fund['eps_growth'] = np.nan

# Treat negative valuation multiples as missing instead of cheap.
fund['earnings_yield'] = np.where(fund['pe'] > 0, 1 / fund['pe'], np.nan)
fund['book_to_price'] = np.where(fund['pb'] > 0, 1 / fund['pb'], np.nan)
fund['ebitda_to_ev'] = np.where(fund['ev_ebitda'] > 0, 1 / fund['ev_ebitda'], np.nan)

# Use only positive market cap as the denominator for FCF yield and log size.
positive_mv = fund['mv'].where(fund['mv'] > 0)
fund['fcf_yield'] = fund['fcf'] / positive_mv
fund['log_mv'] = np.log(positive_mv)

# Shift future fundamentals by firm for the optional forecasting block.
fund['future_roe'] = fund.groupby('ticker')['roe'].shift(-1)
fund['future_revenue_growth'] = fund.groupby('ticker')['revenue_growth'].shift(-1)


In [ ]:
prices['month'] = prices['date'].dt.to_period('M').dt.to_timestamp('M')
monthly = (
    prices.sort_values(['ticker', 'date'])
    .groupby(['ticker', 'month'], as_index = False) # group the sorted data by ticker, momth
    .tail(1)[['ticker', 'month', 'close']] # pick the last price in each month, keeps only the ticker, month, close col
)
all_months = pd.date_range(monthly["month"].min(), monthly["month"].max(), freq="ME")
tickers = monthly["ticker"].drop_duplicates().sort_values()
full_index = pd.MultiIndex.from_product([tickers, all_months], names=["ticker", "month"])

monthly = (
    monthly.set_index(['ticker', 'month'])
    .reindex(full_index)
    .reset_index()
    .sort_values(['ticker', 'month'])
)
monthly

,ticker,month,close
0,AAA,2013-01-31,3.777
1,AAA,2013-02-28,3.804
2,AAA,2013-03-31,3.540
3,AAA,2013-04-30,3.592
4,AAA,2013-05-31,4.121
...,...,...,...
55999,YEG,2025-08-31,14.100
56000,YEG,2025-09-30,14.250
56001,YEG,2025-10-31,12.500
56002,YEG,2025-11-30,12.000


In [ ]:
# Forward-fill monthly prices by ticker after building the full calendar grid.
monthly['close'] = monthly.groupby('ticker')['close'].ffill()

# Compute forward returns from the signal-month close to the future month close.
for horizon in [3, 6, 12]:
    monthly[f'ret_{horizon}m'] = (
        monthly.groupby('ticker')['close'].shift(-horizon) / monthly['close'] - 1
    )

# Compute 12-1 momentum using the prior month and excluding the signal month.
monthly['momentum_12_1'] = (
    monthly.groupby('ticker')['close'].shift(1) / monthly.groupby('ticker')['close'].shift(12) - 1
)

# Pivot monthly close prices to the matrix shape required by vectorbtpro.
price_matrix = monthly.pivot(index='month', columns='ticker', values='close').sort_index()
price_matrix


ticker,AAA,AAM,AAT,ABR,ABS,ABT,ACC,ACG,ACL,ADG,...,VRE,VSC,VSH,VSI,VTB,VTO,VTP,VVS,YBM,YEG
month,,,,,,,,,,,,,,,,,,,,,
2013-01-31,3.777,9.658,NaN,NaN,NaN,14.202,4.133,NaN,3.610,NaN,...,NaN,2.451,6.143,2.094,2.850,1.710,NaN,NaN,NaN,NaN
2013-02-28,3.804,9.697,NaN,NaN,NaN,14.639,4.073,NaN,3.610,NaN,...,NaN,2.617,5.785,2.183,2.680,1.885,NaN,NaN,NaN,NaN
2013-03-31,3.540,9.538,NaN,NaN,NaN,15.258,4.583,NaN,3.657,NaN,...,NaN,2.949,6.041,2.453,2.816,1.745,NaN,NaN,NaN,NaN
2013-04-30,3.592,9.538,NaN,NaN,NaN,14.566,4.359,NaN,3.561,NaN,...,NaN,2.677,6.348,2.243,2.782,1.536,NaN,NaN,NaN,NaN
2013-05-31,4.121,9.142,NaN,NaN,NaN,14.894,5.215,NaN,3.530,NaN,...,NaN,2.865,7.730,1.824,3.563,1.723,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08-31,8.120,7.170,3.58,13.37,3.77,64.000,13.950,35.501,11.686,9.40,...,30.40,30.000,45.770,20.400,9.082,12.150,97.891,34.6,10.847,14.10
2025-09-30,8.200,7.100,3.79,14.95,3.63,68.800,13.750,34.536,12.740,9.45,...,32.10,29.500,45.868,19.632,9.082,11.800,97.100,40.5,10.619,14.25
2025-10-31,8.150,7.000,3.52,12.80,3.00,71.300,13.350,35.115,12.788,9.10,...,33.30,23.200,44.010,19.200,12.900,12.200,115.500,61.2,10.581,12.50


In [ ]:
# Apply a one-month reporting lag and trade from that month-end onward.
# This assumes quarterly financial statements are usable in the month after quarter-end.
panel = fund.copy()
quarter_end_month = fund['quarter'].map({1: 3, 2: 6, 3: 9, 4: 12})
panel['period_end'] = pd.to_datetime(
    {'year': fund['year'], 'month': quarter_end_month, 'day': 1},
    errors='coerce'
) + pd.offsets.MonthEnd(0)
panel['signal_date'] = (
    panel['period_end'] + pd.DateOffset(months=4)
).dt.to_period('M').dt.to_timestamp('M')

# Keep the latest accounting period if multiple rows map to the same signal month.
panel = (
    panel.sort_values(['ticker', 'signal_date', 'period_end'])
    .drop_duplicates(['ticker', 'signal_date'], keep='last')
)

# Attach signal-month prices, forward returns and momentum.
panel = panel.merge(
    monthly,
    left_on=['ticker', 'signal_date'],
    right_on=['ticker', 'month'],
    how='left'
).drop(columns='month')


In [ ]:
# define score components and minimum data requirement
component_groups = {
    "value_score": ["earnings_yield", "book_to_price", "ebitda_to_ev"],
    "quality_score": ["roe", "gross_margin", "net_margin"],
    "growth_score": ["gross_profit_growth", "revenue_growth", "ebt_growth", "eps_growth"],
    "cash_score": ["fcf_yield"]
}
minimum_components = {
    "value_score": 2,
    "quality_score": 1,
    "growth_score": 2,
    "cash_score": 1
}

In [ ]:
fund

,ticker,mv,quarter,year,d_a,ebt_growth,roe,eps,fcff,gross_profit_growth,...,gross_margin,net_margin,eps_growth,earnings_yield,book_to_price,ebitda_to_ev,fcf_yield,log_mv,future_roe,future_revenue_growth
0,AAA,NaN,1,2008,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AAA,NaN,2,2008,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AAA,NaN,3,2008,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.2564,NaN
6,AAA,NaN,4,2008,0.58,NaN,0.2564,NaN,-1.026268e+11,NaN,...,0.2093,0.0921,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,AAA,NaN,1,2009,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61190,YEG,2.186543e+12,4,2024,0.23,1.7211,0.0893,639.45,-1.642559e+11,-0.2826,...,0.1665,0.1195,0.818202,0.056085,0.657895,0.017806,-0.075121,28.413343,0.0865,1.9633
61192,YEG,2.493426e+12,1,2025,0.10,1.0123,0.0865,877.16,-2.808651e+11,1.9238,...,0.1703,0.1143,0.761795,0.067476,0.806452,0.028011,-0.112642,28.544679,0.0886,1.1804
61194,YEG,2.407116e+12,2,2025,0.08,1.7671,0.0886,992.27,-2.268324e+11,2.0196,...,0.1645,0.1131,0.844974,0.079051,0.847458,0.048544,-0.094234,28.509450,0.0715,0.1331
61196,YEG,2.349575e+12,3,2025,0.09,-0.6575,0.0715,806.13,-2.492897e+11,-0.4278,...,0.1362,0.0911,0.425509,0.065789,0.877193,0.042337,-0.106100,28.485256,NaN,NaN


In [ ]:
# Winsorize and percentile-rank every component inside each signal month.
for score_name, columns in component_groups.items():
    rank_columns = []
    for column in columns:
        clean_column = f'{column}_clean'
        rank_column = f'{column}_rank'
        panel[clean_column] = panel.groupby('signal_date')[column].transform(
            lambda s: s.clip(s.quantile(0.01), s.quantile(0.99))
        )
        panel[rank_column] = panel.groupby('signal_date')[clean_column].rank(pct=True)
        rank_columns.append(rank_column)

    # Require the configured minimum number of available inputs per score.
    valid_components = panel[rank_columns].notna().sum(axis=1)
    panel[score_name] = panel[rank_columns].mean(axis=1)
    panel.loc[valid_components < minimum_components[score_name], score_name] = np.nan


In [ ]:
panel

,ticker,mv,quarter,year,d_a,ebt_growth,roe,eps,fcff,gross_profit_growth,...,revenue_growth_clean,revenue_growth_rank,ebt_growth_clean,ebt_growth_rank,eps_growth_clean,eps_growth_rank,growth_score,fcf_yield_clean,fcf_yield_rank,cash_score
0,AAA,NaN,1,2008,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AAA,NaN,2,2008,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AAA,NaN,3,2008,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AAA,NaN,4,2008,0.58,NaN,0.2564,NaN,-1.026268e+11,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AAA,NaN,1,2009,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25915,YEG,2.186543e+12,4,2024,0.23,1.7211,0.0893,639.45,-1.642559e+11,-0.2826,...,1.5204,0.955432,1.7211,0.807799,0.818202,0.817664,0.698149,-0.075121,0.272727,0.272727
25916,YEG,2.493426e+12,1,2025,0.10,1.0123,0.0865,877.16,-2.808651e+11,1.9238,...,1.9633,0.952113,1.0123,0.763380,0.761795,0.828986,0.867106,-0.112642,0.196481,0.196481
25917,YEG,2.407116e+12,2,2025,0.08,1.7671,0.0886,992.27,-2.268324e+11,2.0196,...,1.1804,0.927171,1.7671,0.848739,0.844974,0.835735,0.884704,-0.094234,0.206997,0.206997
25918,YEG,2.349575e+12,3,2025,0.09,-0.6575,0.0715,806.13,-2.492897e+11,-0.4278,...,0.1331,0.564607,-0.6575,0.081461,0.425509,0.691643,0.356197,-0.106100,0.210526,0.210526


In [ ]:
panel['d_a_clean'] = panel.groupby('signal_date')['d_a'].transform(
    lambda s: s.clip(s.quantile(0.01), s.quantile(0.99))
)
panel['balance_score'] = panel.groupby('signal_date')['d_a_clean'].rank(ascending = False, pct = True)

panel


,ticker,mv,quarter,year,d_a,ebt_growth,roe,eps,fcff,gross_profit_growth,...,ebt_growth_clean,ebt_growth_rank,eps_growth_clean,eps_growth_rank,growth_score,fcf_yield_clean,fcf_yield_rank,cash_score,d_a_clean,balance_score
0,AAA,NaN,1,2008,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AAA,NaN,2,2008,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AAA,NaN,3,2008,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AAA,NaN,4,2008,0.58,NaN,0.2564,NaN,-1.026268e+11,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.58,0.077626
4,AAA,NaN,1,2009,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25915,YEG,2.186543e+12,4,2024,0.23,1.7211,0.0893,639.45,-1.642559e+11,-0.2826,...,1.7211,0.807799,0.818202,0.817664,0.698149,-0.075121,0.272727,0.272727,0.23,0.448468
25916,YEG,2.493426e+12,1,2025,0.10,1.0123,0.0865,877.16,-2.808651e+11,1.9238,...,1.0123,0.763380,0.761795,0.828986,0.867106,-0.112642,0.196481,0.196481,0.10,0.735876
25917,YEG,2.407116e+12,2,2025,0.08,1.7671,0.0886,992.27,-2.268324e+11,2.0196,...,1.7671,0.848739,0.844974,0.835735,0.884704,-0.094234,0.206997,0.206997,0.08,0.765449
25918,YEG,2.349575e+12,3,2025,0.09,-0.6575,0.0715,806.13,-2.492897e+11,-0.4278,...,-0.6575,0.081461,0.425509,0.691643,0.356197,-0.106100,0.210526,0.210526,0.09,0.754237


In [ ]:
# Average support components so missing optional components do not mechanically boost the score
support_components = ['quality_score', 'growth_score', 'cash_score', 'balance_score']
panel['support_component_count'] = panel[support_components].notna().sum(axis=1)
panel['support_score'] = panel[support_components].mean(axis=1)
panel.loc[panel['support_component_count'] < 2, 'support_score'] = np.nan
panel['conditioned_value_score'] = panel['value_score'] * panel['support_score']


In [ ]:
# build interactions for cross-sectional regressions
panel['value_quality'] = (panel['value_score'] - 0.5) * (panel['quality_score'] - 0.5)
panel['value_growth'] = (panel['value_score'] - 0.5) * (panel['growth_score'] - 0.5)

score_columns = [
    'value_score', 'quality_score', 'growth_score', 'cash_score', 'balance_score',
    'support_score', 'conditioned_value_score'
]

In [ ]:
fundamental_regressors = [
    "value_score", "quality_score", "growth_score", "cash_score", "balance_score",
    "log_mv", "momentum_12_1"
]
fundamental_targets = ["future_roe", "future_revenue_growth"]
fundamental_regression_rows = []

for target in fundamental_targets:
    regression_data = panel[
        ["ticker", "signal_date", target] + fundamental_regressors
    ].dropna().copy()

    # Demean target and regressors by signal date to control for period effects.
    for column in [target] + fundamental_regressors:
        regression_data[f"{column}_dm"] = (
            regression_data[column]
            - regression_data.groupby("signal_date")[column].transform("mean")
        )

    y = regression_data[f'{target}_dm']
    x = regression_data[[f'{column}_dm' for column in fundamental_regressors]]

    model = sm.OLS(y, x).fit(
        cov_type= 'cluster',
        cov_kwds={'groups': regression_data['ticker']}
    )

    for term in x.columns:
        fundamental_regression_rows.append(
            {
                'target': target,
                'term': term,
                'coef': model.params[term],
                'clustered_t_stat': model.tvalues[term],
                'p_value': model.pvalues[term],
                'obs': model.nobs
            }
        )
fundamental_regression = pd.DataFrame(fundamental_regression_rows)

In [ ]:
fundamental_regression['term'] = fundamental_regression['term'].str.replace(r'_dm$', '', regex=True)

In [ ]:
fundamental_regression

,target,term,coef,clustered_t_stat,p_value,obs
0,future_roe,value_score,0.057280,3.468878,5.226363e-04,13085.0
1,future_roe,quality_score,0.286167,15.290565,8.838020e-53,13085.0
2,future_roe,growth_score,0.062536,9.256621,2.110032e-20,13085.0
3,future_roe,cash_score,0.005988,1.123966,2.610275e-01,13085.0
4,future_roe,balance_score,0.006649,0.626257,5.311462e-01,13085.0
5,future_roe,log_mv,0.006780,2.566637,1.026901e-02,13085.0
6,future_roe,momentum_12_1,0.036032,8.101769,5.416555e-16,13085.0
7,future_revenue_growth,value_score,-6.077410,-2.120641,3.395202e-02,13124.0
8,future_revenue_growth,quality_score,2.003422,0.949153,3.425426e-01,13124.0
9,future_revenue_growth,growth_score,0.498904,0.228803,8.190220e-01,13124.0


In [ ]:
# Convert signal ranks into long-only target weight matrices for vectorbtpro.
strategy_signals = ['value_score', 'conditioned_value_score']
weight_by_signal = {}
holding_rows = []
rebalance_rows = []

for signal in strategy_signals:
    target_weights = pd.DataFrame(np.nan, index=price_matrix.index, columns=price_matrix.columns)
    signal_panel = panel.dropna(subset=[signal, 'close']).copy()

    for signal_date, cross_section in signal_panel.groupby('signal_date'):
        if signal_date not in target_weights.index:
            continue

        # Rebalance only tickers with a usable price on the signal date.
        tradable_tickers = price_matrix.columns[price_matrix.loc[signal_date].notna()]
        cross_section = cross_section[cross_section['ticker'].isin(tradable_tickers)].copy()
        if len(cross_section) < MIN_STOCKS:
            continue

        # Set all tradable tickers to zero so dropped holdings are exited at rebalance dates.
        target_weights.loc[signal_date, tradable_tickers] = 0.0
        cross_section['rank_pct'] = cross_section[signal].rank(pct=True)
        long_group = cross_section[cross_section['rank_pct'] >= TOP_QUANTILE].copy()
        if len(long_group) == 0:
            continue

        # Equal-weight the selected top-quintile basket.
        equal_weight = 1 / len(long_group)
        target_weights.loc[signal_date, long_group['ticker']] = equal_weight

        rebalance_rows.append(
            {
                'signal': signal,
                'signal_date': signal_date,
                'tradable_names': len(cross_section),
                'long_names': len(long_group),
                'gross_target_weight': target_weights.loc[signal_date].fillna(0).abs().sum()
            }
        )
        for row in long_group[['ticker', signal, 'rank_pct']].itertuples(index=False):
            holding_rows.append(
                {
                    'signal': signal,
                    'signal_date': signal_date,
                    'ticker': row.ticker,
                    'score': getattr(row, signal),
                    'rank_pct': row.rank_pct,
                    'target_weight': equal_weight
                }
            )

    weight_by_signal[signal] = target_weights

rebalance_summary = pd.DataFrame(rebalance_rows)
strategy_holdings = pd.DataFrame(holding_rows)

# Start all strategies from the latest first rebalance date for fair comparison
first_rebalance_dates = {
    signal: weights.index[weights.notna().any(axis=1)].min()
    for signal, weights in weight_by_signal.items()
}
common_start = max(first_rebalance_dates.values())
common_end = price_matrix.index.max()

print('First rebalance by signal:', {signal: date.date() for signal, date in first_rebalance_dates.items()})
print('Common start:', common_start.date(), 'Common end:', common_end.date())
print(rebalance_summary.groupby('signal')[['tradable_names', 'long_names']].mean().round(2).to_string())


First rebalance by signal: {'value_score': datetime.date(2013, 1, 31), 'conditioned_value_score': datetime.date(2013, 1, 31)}
Common start: 2013-01-31 Common end: 2025-12-31
                         tradable_names  long_names
signal                                             
conditioned_value_score          286.81       57.98
value_score                      287.15       58.06


In [ ]:
# Run vectorbtpro target-percent portfolio backtests.
performance_rows = []
equity_curves = {}
monthly_return_curves = {}
portfolio_by_signal = {}

for signal, target_weights in weight_by_signal.items():
    close = price_matrix.loc[common_start:common_end].copy()
    weights = target_weights.reindex(close.index).copy()

    # Keep only tickers that receive at least one target order.
    used_columns = weights.notna().any(axis=0)
    close = close.loc[:, used_columns]
    weights = weights.loc[:, used_columns]

    # Let vectorbtpro handle order generation, fees, cash sharing and valuation.
    portfolio = vbt.Portfolio.from_orders(
        close,
        size=weights,
        size_type='targetpercent',
        fees=FEE_RATE,
        init_cash=INIT_CASH,
        cash_sharing=True,
        group_by=True,
        call_seq='auto',
        freq='ME'
    )
    portfolio_by_signal[signal] = portfolio

    # Compute monthly performance metrics from vectorbtpro equity and returns.
    equity = portfolio.value.rename(signal)
    monthly_returns = portfolio.returns.rename(signal)
    clean_returns = monthly_returns.dropna()
    ending_value = equity.iloc[-1]
    total_return = ending_value / INIT_CASH - 1
    annualized_return = (ending_value / INIT_CASH) ** (MONTHS_PER_YEAR / len(clean_returns)) - 1
    annualized_volatility = clean_returns.std(ddof=1) * np.sqrt(MONTHS_PER_YEAR)
    sharpe = np.nan if clean_returns.std(ddof=1) == 0 else clean_returns.mean() / clean_returns.std(ddof=1) * np.sqrt(MONTHS_PER_YEAR)
    drawdown = equity / equity.cummax() - 1
    stats = portfolio.stats()

    performance_rows.append(
        {
            'signal': signal,
            'start': equity.index.min(),
            'end': equity.index.max(),
            'months': len(clean_returns),
            'ending_value': ending_value,
            'total_return': total_return,
            'annualized_return': annualized_return,
            'annualized_volatility': annualized_volatility,
            'sharpe': sharpe,
            'max_drawdown': drawdown.min(),
            'total_orders': stats.get('Total Orders', np.nan),
            'total_fees_paid': stats.get('Total Fees Paid', np.nan),
            'avg_long_names': rebalance_summary.loc[rebalance_summary['signal'].eq(signal), 'long_names'].mean(),
        }
    )
    equity_curves[signal] = equity
    monthly_return_curves[signal] = monthly_returns

strategy_performance = pd.DataFrame(performance_rows)
strategy_equity = pd.concat(equity_curves.values(), axis=1)
strategy_monthly_returns = pd.concat(monthly_return_curves.values(), axis=1)

print(strategy_performance.round(4).to_string(index=False))

# Compare the conditioned strategy against the simple value baseline.
baseline_return = strategy_performance.loc[strategy_performance['signal'].eq('value_score'), 'annualized_return'].iloc[0]
primary_return = strategy_performance.loc[strategy_performance['signal'].eq('conditioned_value_score'), 'annualized_return'].iloc[0]
baseline_sharpe = strategy_performance.loc[strategy_performance['signal'].eq('value_score'), 'sharpe'].iloc[0]
primary_sharpe = strategy_performance.loc[strategy_performance['signal'].eq('conditioned_value_score'), 'sharpe'].iloc[0]

print('Conditioned minus value annualized return:', f'{primary_return - baseline_return:.2%}')
print('Conditioned minus value Sharpe:', f'{primary_sharpe - baseline_sharpe:.4f}')


                 signal      start        end  months  ending_value  total_return  annualized_return  annualized_volatility  sharpe  max_drawdown  total_orders  total_fees_paid  avg_long_names
            value_score 2013-01-31 2025-12-31     156  1.395522e+07       12.9552             0.2248                 0.1939  1.1494       -0.4308          3795      390990.6482         58.0577
conditioned_value_score 2013-01-31 2025-12-31     156  1.764054e+07       16.6405             0.2471                 0.1810  1.3204       -0.3890          3994      566381.1382         57.9808
Conditioned minus value annualized return: 2.23%
Conditioned minus value Sharpe: 0.1710
